In [3]:
%pip install tensorflow

^C
Note: you may need to restart the kernel to use updated packages.


     -------------------------------------- 331.7/331.7 MB 3.0 MB/s eta 0:00:00
     ---------------------------------------- 71.9/71.9 kB 2.0 MB/s eta 0:00:00
     ---------------------------------------- 4.6/4.6 MB 7.4 MB/s eta 0:00:00
     -------------------------------------- 206.3/206.3 kB 6.3 MB/s eta 0:00:00
     -------------------------------------- 135.8/135.8 kB 7.8 MB/s eta 0:00:00
     ---------------------------------------- 2.9/2.9 MB 7.1 MB/s eta 0:00:00
     ---------------------------------------- 5.5/5.5 MB 7.8 MB/s eta 0:00:00
     ---------------------------------------- 57.5/57.5 kB 3.0 MB/s eta 0:00:00
     ---------------------------------------- 64.7/64.7 kB ? eta 0:00:00
     ---------------------------------------- 1.4/1.4 MB 7.4 MB/s eta 0:00:00
     -------------------------------------- 436.9/436.9 kB 3.9 MB/s eta 0:00:00
     ---------------------------------------- 26.4/26.4 MB 7.0 MB/s eta 0:00:00
     ---------------------------------------- 72.5/72.5

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip available: 22.2.2 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Activation, Dropout, Flatten, Dense
from tensorflow.keras import backend as K

ModuleNotFoundError: No module named 'numpy'

In [ ]:
img_width, img_height = 150, 150
base_dir = './teeth_images'
train_data_dir = base_dir

epochs = 20
batch_size = 16

if K.image_data_format() == 'channels_first':
    input_shape = (3, img_width, img_height)
else:
    input_shape = (img_width, img_height, 3)

In [ ]:
model = Sequential()

model.add(Conv2D(32, (3, 3), input_shape=input_shape))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Conv2D(32, (3, 3)))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Conv2D(64, (3, 3)))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Flatten())
model.add(Dense(64))
model.add(Activation('relu'))
model.add(Dropout(0.5))
model.add(Dense(1))
model.add(Activation('sigmoid'))

model.summary()

In [ ]:
from tensorflow.keras.metrics import Recall, Precision

model.compile(loss='binary_crossentropy',
              optimizer='rmsprop',
              metrics=['accuracy', Recall(name='recall'), Precision(name='precision')])

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1. / 255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

validation_datagen = ImageDataGenerator(rescale=1. / 255)

train_generator = train_datagen.flow_from_directory(
    train_data_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='binary',
    subset='training'
)


validation_generator = train_datagen.flow_from_directory(
    train_data_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='binary',
    subset='validation'
)

nb_train_samples = train_generator.samples
nb_validation_samples = validation_generator.samples

print(f"Знайдено {nb_train_samples} зображень для навчання.")
print(f"Знайдено {nb_validation_samples} зображень для валідації.")

In [ ]:
history = model.fit(
    train_generator,
    steps_per_epoch=nb_train_samples // batch_size,
    epochs=epochs,
    validation_data=validation_generator,
    validation_steps=nb_validation_samples // batch_size
)

In [ ]:
model.save_weights('dental_pathology_binary_v1.weights.h5')
print("Ваги моделі збережено.")

In [ ]:
test_data_dir = './test_images'
test_datagen = ImageDataGenerator(rescale=1. / 255)

test_generator = test_datagen.flow_from_directory(
    test_data_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='binary',
    shuffle=False
)
scores = model.evaluate(test_generator)

print("\nРезультати на тестовому наборі")
print(f"Loss: {scores[0]:.4f}")
print(f"Accuracy: {scores[1]*100:.2f}%")
print(f"Recall: {scores[2]*100:.2f}%")
print(f"Precision: {scores[3]*100:.2f}%")

test_recall = scores[2]
test_precision = scores[3]

if (test_precision + test_recall) > 0:
    f1_score = 2 * (test_precision * test_recall) / (test_precision + test_recall)
    print(f"F1-score: {f1_score:.4f}")
else:
    print("Не вдалося розрахувати F1-score (Precision + Recall = 0).")